<a href="https://colab.research.google.com/github/keiseki-eng/E_SHIKAKU/blob/main/%E6%8F%90%E5%87%BA%E7%89%88_Crypto_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 修了課題DEMO③ 仮想通貨

©2026. For information, contact Deloitte Tohmatsu Group.

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [161]:
%%capture
!pip uninstall matplotlib -y
!pip install matplotlib==3.7.1

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.7.0.72

# !pip uninstall torch -y
# !pip install torch==2.0.1

# !pip uninstall torchvision -y
# !pip install torchvision==0.15.2

##はじめに
この修了課題では、仮想通貨の情報を利用して、

未来の仮想通貨の価格予想を行うものです。

今回はDEMOとして、RNN(再帰型ニューラルネットワーク)を用いて学習をさせてみましょう。

##作成までの流れ
大まかな流れとして
1. データのダウンロードと成形

   データダウンロードしてそのまま活用するのは困難です。
   モデルが学習できるよう、カテゴリ変数に置き換えたり、
   欠損値を補完したりなど、成形する必要があります。
2. モデルの構築

   学習を行うモデルのアルゴリズムを理解して、コードを作成します。

3. 学習と結果

   学習を行い結果を確認してみます。★LSTMの採用と隠れ層数の調整で達成
   精度が目標まで達したら、提出用のデータを作成します。

#1.データのダウンロードと成形

## データのダウンロード

In [162]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [163]:
# 学習データのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O' -O crypto_train.csv

# テストデータのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1VhzCcjNSDxGRG86Za653zHHpjVCdSPD3' -O crypto_test_x.csv

--2026-06-12 08:48:57--  https://drive.google.com/uc?export=download&id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O
Resolving drive.google.com (drive.google.com)... 74.125.128.138, 74.125.128.102, 74.125.128.100, ...
Connecting to drive.google.com (drive.google.com)|74.125.128.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O&export=download [following]
--2026-06-12 08:48:58--  https://drive.usercontent.google.com/download?id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.143.132, 2a00:1450:4013:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.143.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24244 (24K) [application/octet-stream]
Saving to: ‘crypto_train.csv’

crypto_train.csv    100%[===================>]  23.68K  --.-KB/s 

In [164]:
# 学習データの確認
train = pd.read_csv('crypto_train.csv', index_col=0)
train

,Mon,Tue,Wed,Thu,Fri,Sat,Sun
0,144.539993,139.000000,116.989998,105.209999,97.750000,112.500000,115.910004
1,112.300003,111.500000,113.566002,112.669998,117.199997,115.242996,115.000000
2,117.980003,111.500000,114.220001,118.760002,123.014999,123.498001,121.989998
3,122.000000,122.879997,123.889000,126.699997,133.199997,131.979996,133.479996
4,129.744995,129.000000,132.300003,128.798996,129.000000,129.300003,122.292000
...,...,...,...,...,...,...,...
186,739.247986,751.346985,744.593994,740.289001,741.648987,735.382019,732.034973
187,735.812988,735.604004,745.690979,756.773987,777.943970,771.155029,773.872009
188,758.700012,764.223999,768.132019,770.809998,772.794006,774.650024,769.731018
189,780.086975,780.556030,781.481018,778.088013,784.906982,790.828979,790.530029


## データの成形

今回使用するRNNのために必要な処理を施していきます。

In [165]:
# 正規化処理
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler = scaler.fit(train.values)

# 正規化する
s_train = scaler.transform(train)
# 元の値に戻す場合
inv_train = scaler.inverse_transform(s_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [166]:
# データの形状確認
print(s_train.shape)

(191, 7)


In [167]:
# シーケンスデータの教師ありデータを作成する
def make_seq(data, seq_len=6):
    X = data[:, 0:seq_len]
    X = np.expand_dims(X, axis=2)
    Y = data[:, seq_len:]
    return X, Y

X, Y = make_seq(s_train)

def train_val_split(X, Y, val_rate):
  rate = int(X.shape[0]*(1-val_rate))
  train_X, train_Y, val_X, val_Y = X[:rate], Y[:rate], X[rate:], Y[rate:]
  return train_X, train_Y, val_X, val_Y

train_X, train_Y, val_X, val_Y = train_val_split(X, Y, 0.1)

In [168]:
# データの形状確認
print('train_X:', train_X.shape)
print('train_Y:', train_Y.shape)

train_X: (171, 6, 1)
train_Y: (171, 1)


# 2.モデルの構築

## 層の設定

In [169]:
# RNN層の設定
"""
RNN
・再帰ニューラルネットワーク
・コンテキストの単語の並び方を逃していることを対策するための層。
・コンテキストの長さによらず、そのコンテキストの情報を記憶するメカニズムをもつ。
・ループする経路（閉じた経路）をもつのが特徴で、そのことで過去の情報を記憶しながら、最新のデータへと更新できる。
"""

class RNN:
    def __init__(self, Wx, Wh, b):
        #引数でわたされたパラメタをメンバ変数paramsにリストとして設定。
        self.params = [Wx, Wh, b]
        #各パラメータに対応する形で勾配gradsを初期化し、格納
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        #逆伝播の計算時に使用する中間データcacheをNoneで初期化
        self.cache = None

    def forward(self, x, h_prev):
        """
        x: (N, D) 現在時刻の入力
        h_prev: (N, H) ひとつ前の隠れ状態
        return: h_next (N, H) 次の隠れ状態
        """
        Wx, Wh, b = self.params

        # RNN基本式: h_next = tanh(x Wx + h_prev Wh + b)
        t = np.dot(h_prev, Wh) + np.dot(x, Wx) + b
        h_next = np.tanh(t)

        # 逆伝播で使うので保持
        self.cache = (x, h_prev, h_next)
        return h_next

    def backward(self, dh_next):
        """
        dh_next: (N, H) 損失Lに対する「次の隠れ状態 h_next」の勾配 dL/dh_next
        return:
          dx      : (N, D) ひとつ前の層への勾配
          dh_prev : (N, H) ひとつ前の時刻の隠れ状態への勾配
        勾配は self.grads にも格納する。
        """
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache

        # tanhの微分: d/dt tanh(t) = 1 - tanh(t)^2
        dt = dh_next * (1 - h_next ** 2)

        # バイアス勾配: 時系列方向に和をとる（バッチ方向Nでsum）
        db = np.sum(dt, axis=0)

        # Wh勾配: h_prev^T dt
        dWh = np.dot(h_prev.T, dt)

        # dh_prev: dt Wh^T
        dh_prev = np.dot(dt, Wh.T)

        # Wx勾配: x^T dt
        dWx = np.dot(x.T, dt)

        # dx: dt Wx^T
        dx = np.dot(dt, Wx.T)

        # 勾配を保存（既存配列にブロードキャスト代入してるのはin-place更新）
        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev

In [170]:
# TimeRNN層の設定
# T個(Tは任意の数)のRNNレイヤから構成される層
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateful=False):#statefulはTimeRNN層の隠れ状態を維持するかしないかを意味する。長い時系列データを維持するときはTrueにする必要がある。
        # RNNセルが使う重みを共有するので、TimeRNNも同じparamsをまとめ保持
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None  # 各時刻tごとにRNNセルを1個ずつ持っておく
        self.h, self.dh = None, None  # hは直前ステップの隠れ状態
        self.stateful = stateful

    def forward(self, xs):
        """
        xs: (N, T, D)
          N: バッチサイズ
          T: 時系列長
          D: 入力次元
        return:
          hs: (N, T, H) 各時刻の隠れ状態を全部並べたもの
        """
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        D, H = Wx.shape

        self.layers = []
        hs = np.empty((N, T, H), dtype='f')#出力用の容器を用意する。

        # stateful=False か、もしくはまだhを持ってない場合はリセット
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')


        # 時系列方向にRNNセルを順に適用していく
        for t in range(T):
            layer = RNN(*self.params)                       # 同じ重みで新しいRNNセルを生成
            self.h = layer.forward(xs[:, t, :], self.h)     # 1ステップ分forward
            hs[:, t, :] = self.h                            # 出力を保存
            self.layers.append(layer)                       # 逆伝播用に保持

        return hs

    def backward(self, dhs):
        """
        dhs: (N, T, H)
          各時刻の隠れ状態 h(t) に対する損失勾配 dL/dh(t)
        return:
          dxs: (N, T, D) 入力列xsに対する勾配
        また self.grads にパラメータの合計勾配を格納し、self.dh に「次バッチへ渡すべき h の勾配」を保持。
        """
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D, H = Wx.shape

        dxs = np.empty((N, T, D), dtype='f')  # 各時刻のdxを格納する入れ物
        dh = 0                                # 次の時刻(将来)から伝わってくる勾配を蓄積
        grads = [0, 0, 0]                     # Wx, Wh, b の勾配を積算するための一時変数

        # 時刻T-1→0へ逆順にバックプロパゲーション（BPTT）
        for t in reversed(range(T)):
            layer = self.layers[t]
            # 将来からの勾配dhと、今回のdhs[:, t, :]を足してbackward
            dx, dh = layer.backward(dhs[:, t, :] + dh)#合算した勾配
            dxs[:, t, :] = dx

            # 各RNNセルの勾配を足しこんで合算
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        # 合算した勾配をメンバにコピーしておく
        for i, grad in enumerate(grads):
            self.grads[i][...] = grad

        # 最初の時刻に対する「前の隠れ状態h_prev」側の勾配を記憶
        self.dh = dh

        return dxs

    def set_state(self, h):
        """
        外から隠れ状態hを手動でセットする。（stateful RNNで続きをやりたいときに使う）
        """
        self.h = h

    def reset_state(self):
        """
        隠れ状態hを捨てる。stateful=Trueで学習していても明示的にリセットできる。
        """
        self.h = None

In [171]:
# Affine層の設定
class Affine:
    """
    通常の全結合層。(N, D) の入力を (N, O) の出力へ線形変換する。
    RNNの最終時刻の隠れ状態から回帰値を出力するのに使う。
    """
    def __init__(self, W, b):
        self.params = [W, b]                 # W: (D, O), b: (O,)
        self.grads = [np.zeros_like(W), np.zeros_like(b)]
        self.x = None                        # forwardで使った入力を保存

    def forward(self, x):
        """
        x: (N, D)
        return: (N, O)
        """
        W, b = self.params
        self.x = x
        return np.dot(x, W) + b

    def backward(self, dout):
        """
        dout: (N, O) = dL/d(out)
        return: dx (N, D)
        さらに self.grads に dW, db を格納
        """
        W, b = self.params

        db = np.sum(dout, axis=0)        # バイアス勾配
        dW = np.dot(self.x.T, dout)      # 重み勾配
        dx = np.dot(dout, W.T)           # 入力側の勾配

        self.grads[0][...] = dW
        self.grads[1][...] = db

        return dx

In [172]:
# MeanSquaredError損失層
class MeanSquaredError:
    """
    平均二乗誤差(MSE)損失層。
    回帰問題なのでクロスエントロピーではなくMSEを使う。
    L = (1 / (2N)) * Σ (y - t)^2
    """
    def __init__(self):
        self.params, self.grads = [], []  # 学習パラメータはなし
        self.cache = None

    def forward(self, y, t):
        """
        y: (N, O) モデルの予測値
        t: (N, O) 正解値
        return: スカラー損失
        """
        N = y.shape[0]
        loss = np.sum((y - t) ** 2) / (2 * N)
        self.cache = (y, t, N)
        return loss

    def backward(self, dout=1):
        """
        dout: 上流からのスカラー勾配(基本1)
        return: dx (N, O) = dL/dy
        """
        y, t, N = self.cache
        dx = dout * (y - t) / N
        return dx

## 最適化手法の設定

In [173]:
#  最適化手法の設定
#  確率的勾配降下法（Stochastic Gradient Descent）
class SGD:
    """
    params[i] -= lr * grads[i]
    を繰り返してパラメータを更新する。
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        # 渡された params と grads のリストを同じインデックスで対応させて更新
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]

## モデルの設定

In [174]:
# モデルSimpleRnnの設定

class SimpleRnn:
    def __init__(self, input_size, hidden_size, output_size):
        D, H, O = input_size, hidden_size, output_size
        rn = np.random.randn  # 正規分布乱数ショートカット

        # 重みの初期化
        # RNN用: 入力→隠れ, 隠れ→隠れ, バイアス
        rnn_Wx = (rn(D, H) / np.sqrt(H)).astype('f')
        rnn_Wh = (rn(H, H) / np.sqrt(H)).astype('f')
        rnn_b = np.zeros(H).astype('f')

        # 出力用Affine: 隠れ→出力 (H×O) とバイアス (O,)
        affine_W = (rn(H, O) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(O).astype('f')

        #レイヤの生成
        self.rnn_layer = TimeRNN(rnn_Wx, rnn_Wh, rnn_b, stateful=False)
        self.affine_layer = Affine(affine_W, affine_b)
        # MSE損失 (回帰版)
        self.loss_layer = MeanSquaredError()

        self.layers = [self.rnn_layer, self.affine_layer]

        # すべてのパラメータと勾配を1つのリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs):
        """
        xs: (N, T, D) 入力系列
        return: (N, O) 予測値
        """
        hs = self.rnn_layer.forward(xs)
        # 最終時刻の隠れ状態のみ使って回帰値を出力する
        h_last = hs[:, -1, :]
        y = self.affine_layer.forward(h_last)
        return y

    def forward(self, xs, ts):
        """
        xs: (N, T, D) 入力系列
        ts: (N, O)    正解値
        return: 損失スカラー
        """
        y = self.predict(xs)
        loss = self.loss_layer.forward(y, ts)
        return loss

    def backward(self, dout=1):
        """
        MeanSquaredError から逆向きに戻っていって
        各層の勾配を self.grads に反映する。
        """
        dy = self.loss_layer.backward(dout)
        dh_last = self.affine_layer.backward(dy)

        # Affineに渡したのは最終時刻の隠れ状態だけなので、
        # TimeRNNに渡す勾配は (N, T, H) の最終時刻にだけdh_lastを入れる
        N, H = dh_last.shape
        T = len(self.rnn_layer.layers)
        dhs = np.zeros((N, T, H), dtype='f')
        dhs[:, -1, :] = dh_last

        self.rnn_layer.backward(dhs)
        return None

## Kerasモデルの構築

In [175]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Keras SimpleRNNモデルの定義
def build_keras_model(input_shape, hidden_size, output_size):
    model = keras.Sequential([
        layers.LSTM(hidden_size, return_sequences=False, input_shape=input_shape),
        layers.Dense(output_size)
    ])
    return model

In [176]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import os

# Ensure dataframes are available. If not found, attempt to download.
try:
    train = pd.read_csv('crypto_train.csv', index_col=0)
except FileNotFoundError:
    print("crypto_train.csv not found, attempting to download...")
    !wget 'https://drive.google.com/uc?export=download&id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O' -O crypto_train.csv
    train = pd.read_csv('crypto_train.csv', index_col=0)

try:
    test_x = pd.read_csv('crypto_test_x.csv', index_col=0)
except FileNotFoundError:
    print("crypto_test_x.csv not found, attempting to download...")
    !wget 'https://drive.google.com/uc?export=download&id=1VhzCcjNSDxGRG86Za653zHHpjVCdSPD3' -O crypto_test_x.csv
    test_x = pd.read_csv('crypto_test_x.csv', index_col=0)

# ターゲット値のMinMaxScalerを準備
# `train`データフレームの最後の列（'Sun'）がターゲット変数であると仮定
target_column_index = train.shape[1] - 1 # 最後の列のインデックス
original_target_values = train.iloc[:, target_column_index].values.reshape(-1, 1)
target_scaler = MinMaxScaler()
target_scaler.fit(original_target_values)

# 入力特徴量 (Mon-Sat) のMinMaxScalerを準備
# `train`データフレームの最後の列を除くすべての列を入力特徴量として使用
input_feature_scaler = MinMaxScaler()
input_feature_scaler.fit(train.iloc[:, :-1].values)


MinMaxScaler()

#3.精度確認から提出まで

##学習

In [177]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Ensure 'train' DataFrame is loaded for scaling
# If 'train' is not defined, we'll try to load it from CSV
# This also handles the case where 'scaler' might not be defined from a previous cell.
try:
    if 'train' not in locals():
        train = pd.read_csv('crypto_train.csv', index_col=0)
except NameError:
    train = pd.read_csv('crypto_train.csv', index_col=0)

# 正規化処理 (Copied from nlWfOIIaMdpK and elGnEnPIe0VK for robustness)
scaler = MinMaxScaler()
scaler = scaler.fit(train.values)
s_train = scaler.transform(train)

# シーケンスデータの教師ありデータを作成する (Copied from elGnEnPIe0VK for robustness)
def make_seq(data, seq_len=6):
    X = data[:, 0:seq_len]
    X = np.expand_dims(X, axis=2)
    Y = data[:, seq_len:]
    return X, Y

X_raw, Y_raw = make_seq(s_train)

def train_val_split(X, Y, val_rate):
  rate = int(X.shape[0]*(1-val_rate))
  train_X, train_Y, val_X, val_Y = X[:rate], Y[:rate], X[rate:], Y[rate:]
  return train_X, train_Y, val_X, val_Y

train_X, train_Y, val_X, val_Y = train_val_split(X_raw, Y_raw, 0.1)


# 学習のハイパーパラメータ
input_size = train_X.shape[2] # 入力データのひとつあたりの次元 (features)
hidden_size = 32             # 隠れ状態の次元 (units for SimpleRNN)
output_size = train_Y.shape[1] # 出力データのひとつあたりの次元
seq_len = train_X.shape[1] # シーケンス長 (timesteps)

epochs = 100        # 学習エポック数
learning_rate = 0.01 # 学習率

# Kerasモデルのビルド
keras_model = build_keras_model(
    input_shape=(seq_len, input_size),
    hidden_size=hidden_size,
    output_size=output_size
)

# モデルのコンパイル
keras_model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss='mean_squared_error')

# 学習データの設定
X = train_X.astype('float32')
Y = train_Y.astype('float32')

# 検証データの設定 (もしあれば)
val_X_keras = val_X.astype('float32')
val_Y_keras = val_Y.astype('float32')

# 学習ループ
history = keras_model.fit(X, Y, epochs=epochs, verbose=0, validation_data=(val_X_keras, val_Y_keras))

# 損失の表示 (Kerasのhistoryから)
loss_list = history.history['loss']
for epoch, loss_value in enumerate(loss_list):
    print(f'epoch: {epoch+1} Loss: {loss_value:.6f}')

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


epoch: 1 Loss: 0.038199
epoch: 2 Loss: 0.011620
epoch: 3 Loss: 0.004436
epoch: 4 Loss: 0.002449
epoch: 5 Loss: 0.001851
epoch: 6 Loss: 0.001375
epoch: 7 Loss: 0.001130
epoch: 8 Loss: 0.001062
epoch: 9 Loss: 0.000886
epoch: 10 Loss: 0.000912
epoch: 11 Loss: 0.000866
epoch: 12 Loss: 0.000847
epoch: 13 Loss: 0.000893
epoch: 14 Loss: 0.000869
epoch: 15 Loss: 0.000848
epoch: 16 Loss: 0.000805
epoch: 17 Loss: 0.000822
epoch: 18 Loss: 0.000870
epoch: 19 Loss: 0.000864
epoch: 20 Loss: 0.000972
epoch: 21 Loss: 0.000778
epoch: 22 Loss: 0.000798
epoch: 23 Loss: 0.000785
epoch: 24 Loss: 0.000952
epoch: 25 Loss: 0.000845
epoch: 26 Loss: 0.000824
epoch: 27 Loss: 0.000735
epoch: 28 Loss: 0.000874
epoch: 29 Loss: 0.000782
epoch: 30 Loss: 0.000763
epoch: 31 Loss: 0.000771
epoch: 32 Loss: 0.000907
epoch: 33 Loss: 0.001018
epoch: 34 Loss: 0.001162
epoch: 35 Loss: 0.000913
epoch: 36 Loss: 0.001231
epoch: 37 Loss: 0.000835
epoch: 38 Loss: 0.000723
epoch: 39 Loss: 0.000698
epoch: 40 Loss: 0.001067
epoch: 41

In [178]:
# 検証データで精度を確認する

# Kerasモデルで予測
pred_keras = keras_model.predict(val_X_keras)

# スケーラーを使用して、標準化された予測データを元のスケールに戻す
# `val_Y_keras`もターゲットスケーラーで逆変換
inverse_val_Y = target_scaler.inverse_transform(val_Y_keras)
inverse_pred = target_scaler.inverse_transform(pred_keras)

# sklearnのmean_squared_error関数を使用してRMSEを計算
from sklearn.metrics import mean_squared_error
import math

# 平均二乗誤差の平方根（RMSE）を計算
rmse = math.sqrt(mean_squared_error(inverse_val_Y, inverse_pred))
print('検証RMSE:', rmse)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
検証RMSE: 21.260934686650067


In [179]:
# plt.plot(Y[:])
# plt.plot(pred[:])

今回の検証RMSEの値を確認すると、このモデルの仕様では目標の精度には達していないことが分かります。

他のライブラリをインポートして、より精度が高くなるようなモデルを構築してみると

精度が向上するかもしれません。

## テストデータに対して予測値を出力する
今回学習したRNNが

テスト用のデータを予測して、結果をcsvファイルとして出力するまでを

掲載してみました。

このcsvファイルを「修了課題提出用サイト」にアップロードすると結果を確認することができます。

In [180]:
# テストデータの確認
test_x = pd.read_csv('crypto_test_x.csv', index_col=0)
test_x

,Mon,Tue,Wed,Thu,Fri,Sat
0,1021.750000,1043.839966,1154.729980,1013.380005,902.200989,908.585022
1,902.828003,907.679016,777.757019,804.833984,823.984009,818.411987
2,831.533997,907.937988,886.617981,899.072998,895.026001,921.789001
3,921.012024,892.687012,901.541992,917.585999,919.750000,921.590027
4,920.382019,970.403015,989.023010,1011.799988,1029.910034,1042.900024
5,1038.150024,1061.349976,1063.069946,994.382996,988.674011,1004.450012
6,990.642029,1004.549988,1007.479980,1027.439941,1046.209961,1054.420044
7,1079.979980,1115.300049,1117.439941,1166.719971,1173.680054,1143.839966
8,1179.969971,1179.969971,1222.500000,1251.010010,1274.989990,1255.150024
9,1272.829956,1223.540039,1150.000000,1188.489990,1116.719971,1175.829956


In [181]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler # Ensure MinMaxScaler is imported here too

# Ensure 'train' DataFrame is loaded to fit the feature scaler
try:
    if 'train' not in locals():
        train = pd.read_csv('crypto_train.csv', index_col=0)
except FileNotFoundError:
    print("crypto_train.csv not found, attempting to download...")
    !wget 'https://drive.google.com/uc?export=download&id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O' -O crypto_train.csv
    train = pd.read_csv('crypto_train.csv', index_col=0)
except NameError:
    # Fallback if 'train' is somehow not in locals() despite FileNotFoundError check
    train = pd.read_csv('crypto_train.csv', index_col=0)

# Ensure 'test_x' DataFrame is loaded for scaling
try:
    if 'test_x' not in locals():
        test_x = pd.read_csv('crypto_test_x.csv', index_col=0)
except FileNotFoundError:
    print("crypto_test_x.csv not found, attempting to download...")
    !wget 'https://drive.google.com/uc?export=download&id=1VhzCcjNSDxGRG86Za653zHHpjVCdSPD3' -O crypto_test_x.csv
    test_x = pd.read_csv('crypto_test_x.csv', index_col=0)
except NameError:
    # Fallback if 'test_x' is somehow not in locals()
    test_x = pd.read_csv('crypto_test_x.csv', index_col=0)


# テストデータも正規化する (Using a scaler fitted specifically on training input features)
# input_feature_scaler is now expected to be defined in 1ab2b613
s_test_x = input_feature_scaler.transform(test_x) # Transform test_x (which also has 6 features)

# データ形式をモデルに対して適正化する
# Keras SimpleRNN expects input shape (batch_size, timesteps, features)
s_test_x = np.expand_dims(s_test_x, axis=2).astype('float32')

# Kerasモデルで予測
# Ensure keras_model is defined (from K5s205cHX-ha)
# Ensure target_scaler is defined (from 1ab2b613)
pred_keras_scaled = keras_model.predict(s_test_x)

# もとに戻した値 (using target_scaler for inverse transformation)
inv_pred = target_scaler.inverse_transform(pred_keras_scaled)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step


In [182]:
# 予測した値を出力する
pred = pd.DataFrame(inv_pred, columns=['Sun'])
pred.to_csv('crypto_pred.csv')
pred

,Sun
0,945.527039
1,834.375549
2,918.410950
3,927.065857
4,1026.612793
5,1004.041809
6,1039.348755
7,1128.626709
8,1213.881958
9,1141.110962
